In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, os

SEED = 42
TARGET = 'Stress_Level'

In [24]:
df = pd.read_csv('Student lifestyle data.csv')
df.sample()

,Student_ID,Age,Gender,Department,CGPA,Sleep_Duration,Study_Hours,Social_Media_Hours,Physical_Activity,Stress_Level
88860,89861,20.0,Male,Science,2.47,7.7,4.1,4.9,93.0,2.0


In [25]:
x = df.drop(columns=['Student_ID', 'Stress_Level'])
y = df['Stress_Level']

In [26]:
y.unique()

array([ 5.,  2.,  3.,  6.,  8.,  4., nan,  7.,  9., 10.])

In [27]:
y.astype(str).value_counts().sort_index()

Stress_Level
10.0        4
2.0     13321
3.0     22229
4.0     22838
5.0     24025
6.0     11428
7.0      2212
8.0      1474
9.0       469
nan      2000
Name: count, dtype: int64

Insight: 
1. Target **Imbalanced**
2. Pesebaran target didominasi oleh class 3.0 - 5.0 dengan class paling minoritas berupa 7.0 - 10.0
3. Terdapat **Missing Value** pada target

Action:
1. Untuk pembagian data harus menggunakan **Strastified Split** agar tiap class memiliki perbandingan yang sama baik di train, val, maupun test

In [29]:
def strastified_split(y, fracs=(0.70, 0.15, 0.15), seed=SEED):
    rng = np.random.default_rng(seed)
    parts = [[], [], []]
    for kelas in np.unique(y):
        idx = rng.permutation(np.where(y == kelas)[0])
        n_tr = int(round(fracs[0] * len(idx)))
        n_va = int(round(fracs[1] * len(idx)))
        parts[0].append(idx[:n_tr])
        parts[1].append(idx[n_tr:n_tr + n_va])
        parts[2].append(idx[n_tr + n_va:])
    return  [rng.permutation(np.concatenate(p)) for p in parts]

idx_tr, idx_va, idx_te = strastified_split(df[TARGET].values)

train_df, val_df, test_df = (df.iloc[idx_tr].copy(), df.iloc[idx_va].copy(), df.iloc[idx_te].copy())

bagian = {
    'semua': df, 'latih (70%)': train_df, 'validasi (15%)': val_df, 'uji (15%)': test_df
}

print(pd.DataFrame({
    'jumlah data': [len(d) for d in bagian.values()],
    'jumlah stres >= 7.0': [(d[TARGET] >= 7.0).sum() for d in bagian.values()],
    '% stres >= 7.0': [round((d[TARGET] >= 7.0).mean() * 100, 2) for d in bagian.values()]
}, index=bagian.keys()))

                jumlah data  jumlah stres >= 7.0  % stres >= 7.0
semua                100000                 4159            4.16
latih (70%)           68601                 2911            4.24
validasi (15%)        14700                  624            4.24
uji (15%)             14699                  624            4.25


In [ ]:
class LinearRegression:
    def __init__(self, lr=0.1, epochs=500):
        self.lr, self.epochs = lr, epochs

    def predict(self, X):
        return X @ self.w + self.b

    def loss(self, X, y): 
        return np.mean((self.predict(X) - y) ** 2)

    def fit(self, X, y):
        """Gradiant Descent"""
        self.w, self.b = np.zeros(X.shape[1]), 0.0
        self.riwayat = []
        for _ in range(self.epochs):
            self.riwayat.append(self.loss(X, y))
            err = self.predict(X) - y
            self.w -= self.lr * 2 * X.T @ err / len(y)
            self.b -= self.lr * 2 * err.mean()
        return self

class StressModel:
    KATEGORI = np.array(['Rendah', 'Sedang', 'Tinggi'])

    def __init__(self, prep, model, batas=(3.5, 5.5)): # CATATAN Batas masih ditentukan secara manual / belum tentu ini batas terbaik
        self.prep, self.model, self.batas = prep, model, list(batas)

    @staticmethod
    def ke_kategori(nilai, batas=(3.5, 5.5)): # CATATAN Batas masih ditentukan secara manual / belum tentu ini batas terbaik
        return StressModel.KATEGORI[np.digitize(nilai, batas)]

    def skor(self, data):
        """Prediksi level stres 0-10."""
        return np.clip(self.model.predict(self.prep.transform(data).values), 0, 10)

    def prediksi(self, data):
        s = self.skor(data)
        return s, self.ke_kategori(s, self.batas)

    def koefisien(self):
        """Bobot dalam satuan asli: efek = w / std, intercept = b - sum(efek * mean)."""
        efek = pd.Series(self.model.w, index=self.prep.kolom) / pd.Series(self.prep.std)[self.prep.kolom]
        intercept = self.model.b - (efek * pd.Series(self.prep.mean)[self.prep.kolom]).sum()
        return float(intercept), efek

    def save(self, path):
        p = self.prep
        isi = {'ambang': p.ambang, 'median': p.median, 'modus': p.modus, 'onehot': p.onehot, 'kolom': p.kolom,
               'mean': p.mean, 'std': p.std, 'w': self.model.w.tolist(), 'b': float(self.model.b), 'batas': self.batas}
        with open(path, 'w') as f:
            json.dump(isi, f, indent=1)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            isi = json.load(f)
        prep = Preprocessor(isi['ambang'])
        for k in ['median', 'modus', 'onehot', 'kolom', 'mean', 'std']:
            setattr(prep, k, isi[k])
        model = LinearRegression()
        model.w, model.b = np.array(isi['w']), isi['b']
        return cls(prep, model, isi['batas'])

In [ ]:
%run model_linreg.py

## B.4 Training: fitur asli vs fitur ambang

In [ ]:
hasil = {}
for nama, ambang in [('M1 fitur asli', False), ('M2 fitur ambang', True)]:
    prep = Preprocessor(ambang=ambang).fit(train)
    model = LinearRegression(lr=0.1, epochs=500).fit(prep.transform(train).values, y_tr)
    hasil[nama] = StressModel(prep, model)
    plt.plot(model.riwayat, label=nama)
plt.yscale('log'); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.legend(); plt.title('Kurva loss'); plt.show()